In [0]:
%pip install \
numpy \
scikit-learn \
umap-learn \
optuna \
joblib \
psutil \
pandas \
tqdm
%pip install hdbscan
%pip install faiss-cpu
%pip install optuna-dashboard

dbutils.library.restartPython()

In [0]:
# ============================================================

# STANDARD LIBRARIES

# ============================================================
 
import os

import glob

import time

from pathlib import Path

from multiprocessing import Manager
 
# ============================================================

# NUMERICAL / DATA

# ============================================================
 
import numpy as np

import pandas as pd

import psutil
 
# ============================================================

# VISUALIZATION

# ============================================================
 
import matplotlib.pyplot as plt
 
import plotly.io as pio

import plotly.graph_objects as go

from plotly.subplots import make_subplots
 
pio.renderers.default = "browser"
 
# ============================================================

# PROGRESS BARS

# ============================================================
 
from tqdm.auto import tqdm
 
# ============================================================

# MACHINE LEARNING

# ============================================================
 
from umap import UMAP
 
from sklearn.cluster import MiniBatchKMeans

from sklearn.metrics import silhouette_score
 
# ============================================================

# HYPERPARAMETER TUNING

# ============================================================
 
import optuna
 
# ============================================================

# OPTIONAL PARALLEL UTILITIES

# ============================================================
 
from joblib import Parallel, delayed
 



import time
import psutil
import os
import shutil
import faiss
import hdbscan

In [0]:
# CONFIG (same as your tag logic)

# ----------------------------

cache_dir = Path("/dbfs/tmp/pftsleep_cache")

encoder_name = "PFTSleep"

num_files = 1229

frequency = 125

win_length = 750

hop_length = 750

max_seq_len_sec = 8 * 3600

def cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec):

    return f"{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}"

tag = cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec)

shard_dir = cache_dir / f"{tag}_shards"

shard_files = sorted(glob.glob(str(shard_dir / "Z_part_*.npy")))

assert len(shard_files) > 0, f"No shards found in {shard_dir}"

print(f"Found {len(shard_files)} shards")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 1) Compute total rows cheaply (mmap_mode avoids loading)

# ----------------------------

D = 512

total_rows = 0

first = np.load(shard_files[0], mmap_mode="r")

print("Example shard shape/dtype:", first.shape, first.dtype)

for f in shard_files:

    total_rows += np.load(f, mmap_mode="r").shape[0]

print(f"Total rows: {total_rows:,}  (expected ~5,899,200)")

print(f"Approx raw size float16: {total_rows * D * 2 / 1e9:.2f} GB")

print(f"Approx raw size float32: {total_rows * D * 4 / 1e9:.2f} GB")

# ----------------------------

# 2) Build a memmap on local NVMe (fast + avoids RAM ceilings)

# ----------------------------

local_dir = Path("/local_disk0/pftsleep_memmap")

local_dir.mkdir(parents=True, exist_ok=True)

mm_path = local_dir / f"X__{tag}__l2norm_f32.memmap"

shape_path = local_dir / f"X__{tag}__shape.txt"

# Create/overwrite memmap file

X_mm = np.memmap(mm_path, dtype=np.float32, mode="w+", shape=(total_rows, D))

# ----------------------------

# 3) Fill memmap sequentially (no vstack, no giant allocations)

# ----------------------------

t0 = time.time()

offset = 0

for f in tqdm(shard_files, desc="Writing memmap", unit="file"):

    shard = np.load(f)  # should be float16

    if shard.dtype != np.float16:

        # still fine; we cast below, but this warns you if storage isn't what you expect

        pass

    n = shard.shape[0]

    X_mm[offset:offset+n, :] = shard.astype(np.float32, copy=False)

    offset += n

X_mm.flush()

t1 = time.time()

with open(shape_path, "w") as s:

    s.write(f"{total_rows},{D}\n")

print(f"\nMemmap written: {mm_path}")

print(f"Write time: {t1 - t0:.2f} sec")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 4) In-place L2 normalize in chunks (still memmap-backed)

# ----------------------------

print("\nNormalizing memmap in chunks...")

chunk_rows = 250_000  # tune if you want (100k–500k is fine)

eps = 1e-8

t2 = time.time()

for start in tqdm(range(0, total_rows, chunk_rows), desc="L2 normalize", unit="chunk"):

    end = min(total_rows, start + chunk_rows)

    block = X_mm[start:end, :]  # view into memmap (does not load everything)

    norms = np.linalg.norm(block, axis=1, keepdims=True)

    block /= np.maximum(norms, eps)

X_mm.flush()

t3 = time.time()

print(f"Normalization time: {t3 - t2:.2f} sec")

print("✅ Memmap X is ready. Use X_mm like a normal array: X_mm[i:j]")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 5) Load your metadata normally (small)

# ----------------------------

night_id = np.load(cache_dir / f"night_id__{tag}.npy")

time_idx = np.load(cache_dir / f"time_idx__{tag}.npy")

zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{tag}.npy")

print("Metadata loaded:", night_id.shape, time_idx.shape, zarr_file_idx.shape)
 

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------

# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")

# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")

# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")

# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})

# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')

# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')

# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)

print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")

# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())

# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)

print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")

# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")

In [0]:
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['NUMBA_NUM_THREADS'] = '16'
os.environ['OPENBLAS_NUM_THREADS'] = '16'

In [0]:
import numpy as np

import pandas as pd

import optuna

import hdbscan

import os

import shutil

import time

import psutil
 
from umap import UMAP

from optuna.exceptions import TrialPruned

from sklearn.metrics import silhouette_score, davies_bouldin_score

from hdbscan.validity import validity_index
 
# Ensure numpy array

night_id = np.asarray(night_id)
 
# Sort indices by night_id

sorted_idx = np.argsort(night_id)

sorted_nights = night_id[sorted_idx]
 
# Find boundaries where night changes

split_points = np.where(np.diff(sorted_nights) != 0)[0] + 1
 
# Split indices into groups

groups = np.split(sorted_idx, split_points)
 
# Map night_id -> row indices

unique_nights = sorted_nights[np.concatenate(([0], split_points))]

night_to_rows = dict(zip(unique_nights, groups))
 
print(f"✅ Built mapping for {len(night_to_rows)} nights")
 
# --------------------------------------------------

# BALANCED NIGHT SAMPLER (unchanged)

# --------------------------------------------------
 
def sample_balanced_rows_by_night(night_to_rows, rng, n_nights, max_rows_per_night):
 
    night_keys = np.array(list(night_to_rows.keys()))
 
    chosen_nights = rng.choice(

        night_keys,

        size=min(n_nights, len(night_keys)),

        replace=False

    )
 
    sampled_rows = []
 
    for nid in chosen_nights:
 
        rows = night_to_rows[nid]
 
        if len(rows) > max_rows_per_night:
 
            rows = rng.choice(rows, size=max_rows_per_night, replace=False)
 
        sampled_rows.append(rows)
 
    return np.concatenate(sampled_rows)
 
 
# --------------------------------------------------

# METRICS

# --------------------------------------------------
 
def compute_metrics(X_emb, labels, clusterer):
 
    mask = labels != -1
 
    if mask.sum() < 500:

        return None
 
    unique_clusters = np.unique(labels[mask])
 
    if len(unique_clusters) < 2:

        return None
 
    noise_fraction = float((labels == -1).mean())
 
    X_core = X_emb[mask]

    y_core = labels[mask]
 
    sample_idx = np.random.choice(len(X_core),

                                 min(20000, len(X_core)),

                                 replace=False)
 
    sil = float(silhouette_score(X_core[sample_idx], y_core[sample_idx]))
 
    db = float(davies_bouldin_score(X_core, y_core))
 
    persistence = (

        float(np.mean(clusterer.cluster_persistence_))

        if len(clusterer.cluster_persistence_) > 0

        else 0.0

    )
 
    probabilities = clusterer.probabilities_[mask]

    mean_prob = float(probabilities.mean())

    sample_idx = np.random.choice(len(X_emb), min(20000, len(X_emb)), replace=False)

    try:

        dbcv = validity_index(X_emb[sample_idx].astype(np.float64), labels[sample_idx])

    except ValueError:

        dbcv = 0.0
 
    _, counts = np.unique(y_core, return_counts=True)
 
    largest_cluster_fraction = float(counts.max() / len(y_core))
 
    small_cluster_fraction = float(np.mean(counts < 20))
 
    return dict(

        silhouette=sil,

        davies_bouldin=db,

        persistence=persistence,

        mean_probability=mean_prob,

        dbcv=dbcv,

        n_clusters=len(unique_clusters),

        noise_fraction=noise_fraction,

        largest_cluster_fraction=largest_cluster_fraction,

        small_cluster_fraction=small_cluster_fraction,

    )
 
 
# --------------------------------------------------

# COMPOSITE SCORE (NEW)

# --------------------------------------------------
 
def composite_score(metrics, trial):
 
    if metrics is None:
        return -1e9
 
    # --------------------------------------------------
    # LEARNABLE WEIGHTS
    # --------------------------------------------------
 
    w_dbcv = trial.suggest_float("w_dbcv", 0.25, 0.55)
    w_persistence = trial.suggest_float("w_persistence", 0.05, 0.30)
    w_probability = trial.suggest_float("w_probability", 0.05, 0.25)
    w_silhouette = trial.suggest_float("w_silhouette", 0.00, 0.20)
    w_noise = trial.suggest_float("w_noise", 0.00, 0.15)
    w_fragmentation = trial.suggest_float("w_fragmentation", 0.00, 0.10)
 
    # normalize weights
 
    total = (
        w_dbcv
        + w_persistence
        + w_probability
        + w_silhouette
        + w_noise
        + w_fragmentation
    )
 
    w_dbcv /= total
    w_persistence /= total
    w_probability /= total
    w_silhouette /= total
    w_noise /= total
    w_fragmentation /= total
 
    # --------------------------------------------------
    # NORMALIZE METRICS
    # --------------------------------------------------
 
    dbcv_norm = (metrics["dbcv"] + 1) / 2
 
    noise_target = 0.25
    noise_score = 1 - abs(metrics["noise_fraction"] - noise_target)
 
    collapse_penalty = max(
        0,
        metrics["largest_cluster_fraction"] - 0.80
    )
 
    fragmentation_penalty = metrics["small_cluster_fraction"]
 
    # --------------------------------------------------
    # FINAL SCORE
    # --------------------------------------------------
 
    score = (
        w_dbcv * dbcv_norm
        + w_persistence * metrics["persistence"]
        + w_probability * metrics["mean_probability"]
        + w_silhouette * metrics["silhouette"]
        + w_noise * noise_score
        - w_fragmentation * fragmentation_penalty
        - 0.10 * collapse_penalty
        - 0.10 * metrics["davies_bouldin"]
    )
 
    # prevent 1-cluster collapse solutions
 
    cluster_penalty = min(metrics["n_clusters"] / 8, 1)
 
    return score * cluster_penalty 
 
# --------------------------------------------------

# OPTUNA OBJECTIVE

# --------------------------------------------------

FREEZE_AFTER_N_TRIALS = 40

WEIGHT_STD_THRESHOLD = 0.015
 
def get_frozen_weights_if_ready(study):
 
    completed = [

        t for t in study.trials

        if t.state.name == "COMPLETE"

        and "w_dbcv" in t.params

    ]
 
    if len(completed) < FREEZE_AFTER_N_TRIALS:

        return None
 
    import numpy as np
 
    weight_matrix = np.array([

        [

            t.params["w_dbcv"],

            t.params["w_persistence"],

            t.params["w_probability"],

            t.params["w_silhouette"],

            t.params["w_noise"],

            t.params["w_fragmentation"],

        ]

        for t in completed[-FREEZE_AFTER_N_TRIALS:]

    ])
 
    weight_std = weight_matrix.std(axis=0)
 
    if np.max(weight_std) < WEIGHT_STD_THRESHOLD:
 
        frozen = weight_matrix.mean(axis=0)
 
        frozen /= frozen.sum()
 
        print("\n🔒 Freezing metric weights:", frozen, "\n")
 
        study.set_user_attr("frozen_weights", frozen.tolist())
 
        return frozen
 
    return None
 

def objective(trial):
 
    t0_trial = time.time()
 
    # ---------------- UMAP ----------------
 
    umap_n_neighbors = int(trial.suggest_float("umap_n_neighbors", 10, 100))
 
    umap_n_components = int(trial.suggest_float("umap_n_components", 5, 40, step=5))
 
    umap_min_dist = trial.suggest_float("umap_min_dist", 0.0, 0.5)
 
    # ---------------- HDBSCAN ----------------
 
    hdb_min_cluster_size = trial.suggest_int("hdb_min_cluster_size", 20, 400)
 
    hdb_min_samples = int(trial.suggest_float("hdb_min_samples", 5, 80))
 
    hdb_cluster_selection_method = trial.suggest_categorical(

        "hdb_cluster_selection_method",

        ["eom", "leaf"]

    )
 
    n_resamples = 3
 
    scores = []
 
    metrics_all = []
 
    for rep in range(n_resamples):
 
        rng = np.random.default_rng(1000 + rep)
 
        row_idx = sample_balanced_rows_by_night(

            night_to_rows,

            rng,

            n_nights=len(night_to_rows),

            max_rows_per_night=500

        )
 
        X_sub = np.asarray(X_mm[row_idx], dtype=np.float64)
 
        # ---------------- UMAP ----------------
 
        reducer = UMAP(

            n_neighbors=umap_n_neighbors,

            n_components=umap_n_components,

            min_dist=umap_min_dist,

            metric="cosine",

            random_state=None,

            transform_seed=rep,

            low_memory=True,

            n_jobs=-1,

        )
 
        X_umap = reducer.fit_transform(X_sub).astype(np.float64)
 
        # ---------------- HDBSCAN ----------------
 
        clusterer = hdbscan.HDBSCAN(

            min_cluster_size=hdb_min_cluster_size,

            min_samples=hdb_min_samples,

            metric="euclidean",

            cluster_selection_method=hdb_cluster_selection_method,

            core_dist_n_jobs=16

        )
 
        labels = clusterer.fit_predict(X_umap)
 
        metrics = compute_metrics(X_umap, labels, clusterer)
 
        score = composite_score(metrics, trial)
 
        scores.append(score)
 
        metrics_all.append(metrics)
 
        trial.report(score, step=rep)
 
        # pruning guards
 
        if metrics is None:

            raise TrialPruned()
 
        if metrics["noise_fraction"] > 0.85:

            raise TrialPruned()
 
        if metrics["n_clusters"] < 2:

            raise TrialPruned()
 
        if trial.should_prune():

            raise TrialPruned()
 
    scores = np.array(scores)
 
    valid_metrics = [m for m in metrics_all if m is not None]
 
    if len(valid_metrics) == 0:

        return -1e9
 
    final_score = float(np.mean(scores) - 0.5 * np.std(scores))
 
    trial.set_user_attr("score_mean", float(np.mean(scores)))
 
    trial.set_user_attr("score_std", float(np.std(scores)))
 
    trial.set_user_attr(

        "mean_clusters",

        float(np.mean([m["n_clusters"] for m in valid_metrics]))

    )
 
    trial.set_user_attr(

        "mean_noise",

        float(np.mean([m["noise_fraction"] for m in valid_metrics]))

    )
 
    trial.set_user_attr(

        "mean_dbcv",

        float(np.mean([m["dbcv"] for m in valid_metrics]))

    )
 
    trial.set_user_attr(

        "mean_persistence",

        float(np.mean([m["persistence"] for m in valid_metrics]))

    )
    trial.set_user_attr("w_dbcv", trial.params["w_dbcv"])
    trial.set_user_attr("w_persistence", trial.params["w_persistence"])
    trial.set_user_attr("w_probability", trial.params["w_probability"])
    trial.set_user_attr("w_silhouette", trial.params["w_silhouette"])
    trial.set_user_attr("w_noise", trial.params["w_noise"])
    trial.set_user_attr("w_fragmentation", trial.params["w_fragmentation"])
    t1_trial = time.time()
 
    trial_total = t1_trial - t0_trial
 
    print(

        f"[Trial {trial.number}] "

        f"score={final_score:.4f} | "

        f"clusters={trial.user_attrs['mean_clusters']:.1f} | "

        f"noise={trial.user_attrs['mean_noise']:.2f} | "

        f"dbcv={trial.user_attrs['mean_dbcv']:.3f} | "

        f"time={trial_total/60:.2f}m"

    )
 
    print(

        f"RAM available: "

        f"{psutil.virtual_memory().available/1e9:.1f} GB"

    )
    # --------------------------------------------------

    # PER-TRIAL CSV LOGGING (FULL METRICS EXPORT)

    # --------------------------------------------------
 
    cluster_counts = [m["n_clusters"] for m in valid_metrics]

    noise_vals = [m["noise_fraction"] for m in valid_metrics]

    dbcv_vals = [m["dbcv"] for m in valid_metrics]

    persist_vals = [m["persistence"] for m in valid_metrics]

    prob_vals = [m["mean_probability"] for m in valid_metrics]

    sil_vals = [m["silhouette"] for m in valid_metrics]

    db_vals = [m["davies_bouldin"] for m in valid_metrics]

    largest_vals = [m["largest_cluster_fraction"] for m in valid_metrics]

    small_vals = [m["small_cluster_fraction"] for m in valid_metrics]
 
    trial_record = {
 
        # trial info

        "trial": trial.number,

        "score": final_score,
 
        # UMAP params

        "umap_n_neighbors": umap_n_neighbors,

        "umap_n_components": umap_n_components,

        "umap_min_dist": umap_min_dist,
 
        # HDBSCAN params

        "hdb_min_cluster_size": hdb_min_cluster_size,

        "hdb_min_samples": hdb_min_samples,

        "hdb_cluster_selection_method": hdb_cluster_selection_method,
 
        # metric means

        "mean_n_clusters": np.mean(cluster_counts),

        "mean_noise_fraction": np.mean(noise_vals),

        "mean_dbcv": np.mean(dbcv_vals),

        "mean_persistence": np.mean(persist_vals),

        "mean_probability": np.mean(prob_vals),

        "mean_silhouette": np.mean(sil_vals),

        "mean_davies_bouldin": np.mean(db_vals),

        "mean_largest_cluster_fraction": np.mean(largest_vals),

        "mean_small_cluster_fraction": np.mean(small_vals),
 
        # metric variability (important for stability)

        "std_n_clusters": np.std(cluster_counts),

        "std_noise_fraction": np.std(noise_vals),

        "std_dbcv": np.std(dbcv_vals),

        "std_persistence": np.std(persist_vals),

        "std_probability": np.std(prob_vals),

        "std_silhouette": np.std(sil_vals),
 
        # learned weights

        "w_dbcv": trial.params.get("w_dbcv"),

        "w_persistence": trial.params.get("w_persistence"),

        "w_probability": trial.params.get("w_probability"),

        "w_silhouette": trial.params.get("w_silhouette"),

        "w_noise": trial.params.get("w_noise"),

        "w_fragmentation": trial.params.get("w_fragmentation"),
 
        # runtime

        "trial_total_sec": trial_total,
 
        # frozen scoring weights snapshot

        "frozen_weights": trial.study.user_attrs.get("frozen_weights"),

    }
 
    try:
 
        local_csv = "/local_disk0/umap_hdbscan_joint_trials.csv"
 
        pd.DataFrame([trial_record]).to_csv(
        local_csv,
        mode="a",
        header=not os.path.exists(local_csv),
        index=False
        )
        shutil.copy(local_csv, "/dbfs/tmp/umap_hdbscan_joint_trials.csv")
 
    except Exception as e:
 
        print(f"CSV logging failed: {e}")
 
    return final_score
 
 
# --------------------------------------------------

# STUDY

# --------------------------------------------------
 
study = optuna.create_study(

    study_name="umap_hdbscan_joint_density_v3",

    storage="sqlite:////local_disk0/umap_hdbscan_optuna.db",

    load_if_exists=True,

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=42),

    pruner=optuna.pruners.MedianPruner(

        n_startup_trials=3,

        n_warmup_steps=1,

        interval_steps=1

    )

)
 
study.optimize(objective, n_trials=40)

In [0]:
import numpy as np
import optuna
import hdbscan
from umap import UMAP
from optuna.exceptions import TrialPruned 
 
from sklearn.metrics import silhouette_score, davies_bouldin_score
 
import numpy as np
 
# Ensure numpy array

night_id = np.asarray(night_id)
 
# Sort indices by night_id

sorted_idx = np.argsort(night_id)

sorted_nights = night_id[sorted_idx]
 
# Find boundaries where night changes

split_points = np.where(np.diff(sorted_nights) != 0)[0] + 1
 
# Split indices into groups

groups = np.split(sorted_idx, split_points)
 
# Map night_id -> row indices

unique_nights = sorted_nights[np.concatenate(([0], split_points))]

night_to_rows = dict(zip(unique_nights, groups))
 
print(f"✅ Built mapping for {len(night_to_rows)} nights")
 
 
def sample_balanced_rows_by_night(night_to_rows, rng, n_nights, max_rows_per_night):
    night_keys = np.array(list(night_to_rows.keys()))
    chosen_nights = rng.choice(
        night_keys,
        size=min(n_nights, len(night_keys)),
        replace=False
    )
 
    sampled_rows = []
    for nid in chosen_nights:
        rows = night_to_rows[nid]
        if len(rows) > max_rows_per_night:
            rows = rng.choice(rows, size=max_rows_per_night, replace=False)
        sampled_rows.append(rows)
 
    return np.concatenate(sampled_rows)
 
 
def compute_metrics(X_emb, labels, clusterer):
    mask = labels != -1
    clustered_fraction = float(mask.mean())
    noise_fraction = 1.0 - clustered_fraction
 
    if mask.sum() < 1000:
        return None
 
    unique_clusters = np.unique(labels[mask])
    n_clusters = len(unique_clusters)
 
    if n_clusters < 2:
        return None
 
    X_core = X_emb[mask]
    y_core = labels[mask]
 
    sample_idx = np.random.choice(len(X_core), min(20000, len(X_core)), replace=False)
    sil = float(silhouette_score(X_core[sample_idx], y_core[sample_idx]))
    db = float(davies_bouldin_score(X_core, y_core))
 
    persistence = (
        float(np.mean(clusterer.cluster_persistence_))
        if hasattr(clusterer, "cluster_persistence_") and len(clusterer.cluster_persistence_) > 0
        else 0.0
    )
 
    _, counts = np.unique(y_core, return_counts=True)
    largest_cluster_fraction = float(counts.max() / len(y_core))
    small_cluster_fraction = float(np.mean(counts < 20))
 
    return {
        "silhouette": sil,
        "davies_bouldin": db,
        "persistence": persistence,
        "n_clusters": int(n_clusters),
        "noise_fraction": noise_fraction,
        "largest_cluster_fraction": largest_cluster_fraction,
        "small_cluster_fraction": small_cluster_fraction,
    }
 
 
def composite_score(metrics):
    if metrics is None:
        return -1e9
 
    score = (
        1.00 * metrics["silhouette"]
        - 0.25 * metrics["davies_bouldin"]
        + 0.50 * metrics["persistence"]
        - 0.50 * metrics["noise_fraction"]
        - 0.25 * max(0.0, metrics["largest_cluster_fraction"] - 0.80)
        - 0.25 * metrics["small_cluster_fraction"]
    )
 
    # Optional soft penalty for absurd fragmentation
    if metrics["n_clusters"] > 200:
        score -= 0.01 * (metrics["n_clusters"] - 200)
 
    return float(score)
 
    t0_trial = time.time()

def objective(trial):
    t0_trial = time.time()
    # UMAP
    umap_n_neighbors = trial.suggest_int("umap_n_neighbors", 10, 100, step=5)
    umap_n_components = trial.suggest_int("umap_n_components", 5, 30, step=2)
    umap_min_dist = trial.suggest_float("umap_min_dist", 0.0, 0.5)
 
    # HDBSCAN
    hdb_min_cluster_size = trial.suggest_int("hdb_min_cluster_size", 20, 500, step=20)
    hdb_min_samples = trial.suggest_int("hdb_min_samples", 0, 100, step=5)
    hdb_cluster_selection_method = trial.suggest_categorical("hdb_cluster_selection_method", ["eom", "leaf"])
 
    # Sampling protocol
    n_resamples = 3
    scores = []

    metrics_all = []
 
    for rep in range(n_resamples):

        rng = np.random.default_rng(1000 + rep)
 
        row_idx = sample_balanced_rows_by_night(

            night_to_rows=night_to_rows,

            rng=rng,

            n_nights=len(night_to_rows),

            max_rows_per_night=500,

        )
 
        X_sub = np.asarray(X_mm[row_idx], dtype=np.float32)
 
        # ---------------- UMAP ----------------

        t0_umap = time.time()
 
        reducer = UMAP(

            n_neighbors=umap_n_neighbors,

            n_components=umap_n_components,

            min_dist=umap_min_dist,

            metric="cosine",

            random_state=None,

            transform_seed=rep,

            low_memory=True,

            n_jobs=-1,
            
            verbose=True,

        )
 
        X_umap = reducer.fit_transform(X_sub)

        umap_time = time.time() - t0_umap
 
        # ---------------- HDBSCAN ----------------

        t0_hdb = time.time()
 
        clusterer = hdbscan.HDBSCAN(

            min_cluster_size=hdb_min_cluster_size,

            min_samples=hdb_min_samples,

            cluster_selection_method=hdb_cluster_selection_method,

            metric="euclidean",

            core_dist_n_jobs=16,

        )
 
        labels = clusterer.fit_predict(X_umap)

        hdbscan_time = time.time() - t0_hdb
 
        # ---------------- METRICS ----------------

        metrics = compute_metrics(X_umap, labels, clusterer)

        score = composite_score(metrics)
 
        scores.append(score)

        metrics_all.append(metrics)
 
        # ---------------- EARLY STOPPING ----------------

        # Report intermediate result to Optuna

        trial.report(score, step=rep)
 
        # Hard rejection conditions

        if metrics is None:

            print(f"Trial {trial.number} pruned (invalid clustering)")

            raise TrialPruned()
 
        if metrics["noise_fraction"] > 0.85:

            print(f"Trial {trial.number} pruned (too much noise)")

            raise TrialPruned()
 
        if metrics["n_clusters"] < 2:

            print(f"Trial {trial.number} pruned (too few clusters)")

            raise TrialPruned()
 
        # Optuna pruning (learned pruning)

        if trial.should_prune():

            print(f"Trial {trial.number} pruned by Optuna at step {rep}")

            raise TrialPruned()
 
 
    scores = np.array(scores, dtype=float)
    valid_metrics = [m for m in metrics_all if m is not None]
 
    if len(valid_metrics) == 0:
        return -1e9
    t1_trial = time.time()
    trial_total = t1_trial - t0_trial
    final_score = float(np.mean(scores) - 0.5 * np.std(scores))
 
    trial.set_user_attr("score_mean", float(np.mean(scores)))
    trial.set_user_attr("score_std", float(np.std(scores)))
    trial.set_user_attr("mean_n_clusters", float(np.mean([m["n_clusters"] for m in valid_metrics])))
    trial.set_user_attr("mean_noise_fraction", float(np.mean([m["noise_fraction"] for m in valid_metrics])))
    trial.set_user_attr("mean_silhouette", float(np.mean([m["silhouette"] for m in valid_metrics])))
    trial.set_user_attr("mean_persistence", float(np.mean([m["persistence"] for m in valid_metrics])))
    # -------------------------------------------------

    # SAVE TRIAL RESULTS

    # -------------------------------------------------
 
    local_csv = "/local_disk0/umap_hdbscan_joint_trials.csv"

    dbfs_csv = "/dbfs/tmp/umap_hdbscan_joint_trials.csv"
 
    # Use last valid metrics (or fallback)

    if valid_metrics:
 
        m = {
 
            "n_clusters": int(np.mean([x["n_clusters"] for x in valid_metrics])),
 
            "noise_fraction": float(np.mean([x["noise_fraction"] for x in valid_metrics])),
 
            "silhouette": float(np.mean([x["silhouette"] for x in valid_metrics])),
 
            "davies_bouldin": float(np.mean([x["davies_bouldin"] for x in valid_metrics])),
 
            "persistence": float(np.mean([x["persistence"] for x in valid_metrics])),
 
        }
 
        m_std = {
 
            "n_clusters_std": float(np.std([x["n_clusters"] for x in valid_metrics])),
 
            "noise_fraction_std": float(np.std([x["noise_fraction"] for x in valid_metrics])),
 
            "silhouette_std": float(np.std([x["silhouette"] for x in valid_metrics])),
 
            "persistence_std": float(np.std([x["persistence"] for x in valid_metrics])),
 
        }
 
    else:
 
        m = {
 
            "n_clusters": -1,
 
            "noise_fraction": 1.0,
 
            "silhouette": -1.0,
 
            "davies_bouldin": 999.0,
 
            "persistence": 0.0,
 
        }
 
        m_std = {
 
            "n_clusters_std": 0.0,
 
            "noise_fraction_std": 0.0,
 
            "silhouette_std": 0.0,
 
            "persistence_std": 0.0,
 
        }
 
 
    # ✅ DEFINE trial_record OUTSIDE the if/else block
 
    trial_record = {
 
        "trial": trial.number,
 
        "score": final_score,
 
        # UMAP params
 
        "umap_n_neighbors": umap_n_neighbors,
 
        "umap_n_components": umap_n_components,
 
        "umap_min_dist": umap_min_dist,
 
        # HDBSCAN params
 
        "hdb_min_cluster_size": hdb_min_cluster_size,
 
        "hdb_min_samples": hdb_min_samples,
 
        "hdb_cluster_selection_method": hdb_cluster_selection_method,
 
        # mean metrics
 
        "n_clusters": m["n_clusters"],
 
        "noise_fraction": m["noise_fraction"],
 
        "silhouette": m["silhouette"],
 
        "davies_bouldin": m["davies_bouldin"],
 
        "persistence": m["persistence"],
 
        # variability
 
        "n_clusters_std": m_std["n_clusters_std"],
 
        "noise_fraction_std": m_std["noise_fraction_std"],
 
        "silhouette_std": m_std["silhouette_std"],
 
        "persistence_std": m_std["persistence_std"],
 
        # timing
 
        "umap_time_sec": umap_time,
 
        "hdbscan_time_sec": hdbscan_time,
 
        "trial_total_sec": trial_total,
            # learned metric weights
 
        "w_dbcv": trial.params.get("w_dbcv"),

        "w_persistence": trial.params.get("w_persistence"),

        "w_probability": trial.params.get("w_probability"),

        "w_silhouette": trial.params.get("w_silhouette"),

        "w_noise": trial.params.get("w_noise"),

        "w_fragmentation": trial.params.get("w_fragmentation"),
 
        # frozen weights snapshot (if activated)
 
        "frozen_weights": trial.study.user_attrs.get("frozen_weights"),
 
 
    }
    trial_record["frozen_weights"] = trial.study.user_attrs.get("frozen_weights")
    df = pd.DataFrame([trial_record])
 
    if os.path.exists(local_csv):

        df.to_csv(local_csv, mode="a", header=False, index=False)

    else:

        df.to_csv(local_csv, index=False)
 
    shutil.copy(local_csv, dbfs_csv)
 
    try:
        best_so_far = study.best_value
    except ValueError:
        best_so_far = None
 
    print(

        f"[Trial {trial.number:03d}] "

        f"score={final_score:.4f} | "

        f"best={best_so_far:.4f}" if best_so_far is not None else "best=NA"

        f"clusters={m['n_clusters']} | "

        f"noise={m['noise_fraction']:.2f} | "

        f"time={trial_total/60:.2f}m"

    )
 
 
    print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB")
    
    return final_score
study = optuna.create_study(
    study_name="umap_hdbscan_faiss_cpu_2",
    storage='sqlite:////local_disk0/umap_hdbscan_optuna.db',
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
 
    # 👇 ADD THIS
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=3,   # don't prune first few
        n_warmup_steps=1,     # wait at least 1 resample
        interval_steps=1
    )
)

study.optimize(objective, n_trials=80)

if "frozen_weights" in study.user_attrs:
    best_weights = study.user_attrs["frozen_weights"]
    print("\nFinal learned scoring weights:\n", best_weights)
else:
    print("\nWeights never froze during optimization.")